# GraphRAG 2-Stage vs GraphRAG 1-Stage Text-to-SQL — Evaluation Pipeline (Kaggle)

Notebook ini menjalankan **dua pipeline Text-to-SQL** pada Spider dev set, lalu mengevaluasinya dengan lapisan metrik skripsi. Semua logika ada di `src/` — notebook ini memanggilnya langkah demi langkah dan **menjelaskan apa yang terjadi di tiap langkah**, jadi bisa dibaca sebagai dokumentasi cara kerja pipeline.

## Dua pipeline yang dibandingkan

| Tahap | GraphRAG 2-Stage (`pipeline.py` → `run_single`) | GraphRAG 1-Stage (`baseline.py` → `run_single_baseline`) |
|---|---|---|
| Input | pertanyaan | pertanyaan |
| Pilih tabel | Stage 1: **tabel** dengan skor ≥ threshold tabel (n-gram + cosine BGE-M3, teks embed = nama tabel) | **sama persis** dengan Stage 1 GraphRAG 2-Stage (teks embed = nama tabel). Varian "1-Stage+": `baseline_embed_columns=True` memakai `table <nama> columns <kolom...>` |
| Pilih kolom | Stage 2: **kolom** di tabel kandidat dengan skor ≥ threshold kolom, + kolom dari **value linking** (nilai sel yang disebut di pertanyaan) | — (tidak ada) |
| Jalur JOIN | Graph traversal: shortest path antar **kolom**; yang masuk prompt hanya **kolom JOIN** (`fk_only`) | Table path tracing: shortest path antar **tabel** via FK |
| Schema context | DDL hanya **kolom terpilih** | DDL **semua kolom** tabel terpilih (tanpa pruning) |
| Few-shot | top-k contoh dari `train_spider.json`, dipilih by **skeleton** pertanyaan | **sama persis** (`cfg.few_shot_k`, index yang sama) |
| Prompt → LLM → cleaner | Prompt builder → Qwen2.5-Coder (4-bit, greedy) → SQL cleaner | **sama persis** untuk keduanya |

Kedua pipeline memakai LLM dan few-shot yang **sama**; yang berbeda hanya retrieval-nya.

GraphRAG 2-Stage memakai **preprocessing tetap** hasil trial SLA (trial SLA G7; efek tiap langkah di `notebooks/trial_preprocessing.ipynb`); yang bisa dipilih hanya threshold tabel dan kolom (sel 3). Few-shot dipilih dengan **skeleton** pertanyaan (`src/skeleton/`, `few_shot_selection="skeleton"`).

## Tiga tahap ablation (keputusan `IMPLEMENTATION_DECISIONS.md` poin 28)

| Tahap | Yang dicari | Grid (`config.py`) | Dipilih dengan | Jalankan |
|---|---|---|---|---|
| 1 | threshold tabel & kolom (dipakai kedua pipeline untuk tabel; kolom hanya GraphRAG 2-Stage) | grid di `trial_thresholds.ipynb` | **SLA** (tanpa LLM) | `trial_thresholds.ipynb` |
| 2 | few-shot k (sama untuk kedua pipeline) | `few_shot_frame` | **EX** | `ablation.py --mode {graphrag,baseline}` |
| 3 | model LLM | `llm_model_frame` | EX (+ token) | `MODEL` di Bagian 3, atau `pipeline.py --baseline --models all` |

Notebook ini menjalankan **satu** konfigurasi (satu model) dari ujung ke ujung; tahap 2 dan 3 lebih praktis lewat CLI karena butuh banyak run LLM.

## Alur notebook ini

| Bagian | Isi |
|---|---|
| 1-2 | Setup dan import |
| 3 | **Pilih `MODEL`**: satu-satunya yang diganti tiap orang |
| 4-8 | Baca data, bangun graph, muat model, buat index, siapkan evaluator |
| 9-10 | Contoh proses satu pertanyaan (boleh dilewati) |
| 11-12 | Jalankan 1-Stage dan 2-Stage, lalu amankan prediksi dan `raw_logs` |
| 13 | **Hasil 6 metrik**: satu tabel ringkasan, lalu rincian per metrik |
| 14 | Pembanding: evaluator resmi Spider |
| 15 | **Analisis 6 dimensi** |
| 16 | **Simpan semua hasil** ke JSON dan Excel |

Logika detail ada di `src/experiments/notebook_steps.py`, jadi sel di notebook ini tetap pendek. **GraphRAG 1-Stage** adalah `baseline` di kode, dan **GraphRAG 2-Stage** adalah `graphrag` di kode.

## Cara memakai untuk tiga model

Tiap orang hanya mengganti `MODEL` di sel Bagian 3 (`"3B"`, `"7B"`, atau `"14B"`), lalu menjalankan semua sel. Notebook selalu menjalankan **GraphRAG 1-Stage** (`baseline` di kode) dan **GraphRAG 2-Stage** (`graphrag` di kode). Hasil tiap model masuk ke folder sendiri, lalu digabung dengan `combine_models.py` (Bagian 17).

Semua hasil ditampilkan sebagai tabel angka (`show()` dari `src/experiments/result_tables.py`, modul yang sama dengan export Excel). Notebook ini **tidak menulis interpretasi atau kesimpulan**; pembacaan hasil dilakukan manual.

> **Catatan run final:** untuk skripsi, set `SAMPLE_RATIO = 1.0` (full 1034 query). Data QVT (`build_qvt_variations.py`) menolak `raw_logs` hasil subsampling, karena `query_id` = posisi baris di `dev.json`.

In [1]:
# ============================================================
# 1. SETUP — clone repo, install dependency, cek evaluator Spider
# Sel ini aman dijalankan ulang (semua langkah dicek dulu).
# ============================================================
# os untuk pindah folder, subprocess untuk menjalankan perintah terminal
import os, subprocess
# Path untuk menulis lokasi file dengan aman di semua sistem operasi
from pathlib import Path

# Alamat repo yang di-clone saat notebook jalan di Kaggle
REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
# Branch repo yang dipakai
REPO_BRANCH = "main"

# Fungsi pembantu untuk menjalankan satu perintah terminal
def sh(cmd: str) -> None:
    # Tampilkan perintahnya supaya terlihat apa yang sedang dijalankan
    print(f"$ {cmd}")
    # Jalankan perintah; check=True membuat notebook berhenti kalau perintahnya gagal
    subprocess.run(cmd, shell=True, check=True)

# Fungsi pembantu untuk mengecek apakah sebuah folder adalah root repo ini
def is_repo_root(p: Path) -> bool:
    # Cek file milik repo ini, BUKAN cuma folder "src": Kaggle punya /kaggle/src sendiri,
    # jadi cek "../src" dari /kaggle/working salah mengira /kaggle sebagai root repo.
    return (p / "src" / "core" / "config.py").is_file()

# 1) cwd HARUS root repo: pipeline memakai path relatif (outputs/, data/raw_logs/, external/spider_eval/)
# Kalau folder sekarang bukan root repo, cari atau siapkan dulu
if not is_repo_root(Path.cwd()):
    # Kasus pertama: folder induknya adalah root repo
    if is_repo_root(Path.cwd().parent):  # notebook dijalankan dari notebooks/
        # Naik satu folder ke root repo
        os.chdir("..")
    # Kasus kedua: repo berada di subfolder atau belum ada
    else:                                # Kaggle: repo belum ada di /kaggle/working
        # Repo belum pernah di-clone
        if not is_repo_root(Path("thesis-2803").resolve()):
            # Clone repo dari GitHub
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        # Repo sudah ada dari run sebelumnya
        else:                                # sudah pernah di-clone: pastikan di branch yang benar
            # Ambil versi terbaru dari branch yang dipilih
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        # Masuk ke folder repo
        os.chdir("thesis-2803")
# Simpan lokasi root repo untuk dipakai sel berikutnya
REPO_ROOT = Path.cwd()
# Hentikan notebook kalau ternyata masih bukan root repo
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803 (src/core/config.py tidak ada)"
# Tampilkan lokasi repo
print("Repo root:", REPO_ROOT)
# Tampilkan branch yang sedang aktif
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# 2) dependency (torch, pandas, networkx, tqdm sudah ada di Kaggle -> jangan di-upgrade;
#    `-U pandas` sebelumnya menarik pandas 3.x dan bentrok dengan dependency image Kaggle).
#    nltk dibutuhkan process_sql.py milik Spider.
# Install library yang belum tersedia di Kaggle
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk")

# 3) evaluator resmi Spider sudah ter-commit di external/spider_eval/ (JANGAN diedit) -> cukup dicek ada
# Periksa kedua file evaluator resmi satu per satu
for fname in ("evaluation.py", "process_sql.py"):
    # Hentikan notebook kalau filenya tidak ada
    assert (REPO_ROOT / "external" / "spider_eval" / fname).exists(), f"external/spider_eval/{fname} tidak ada di repo"

# 4) tokenizer data untuk parser SQL Spider
# nltk dipakai parser Spider untuk memecah SQL menjadi token
import nltk
# Unduh dua paket data tokenizer yang dibutuhkan
for pkg in ("punkt", "punkt_tab"):
    # Coba unduh paketnya
    try:
        # quiet=True supaya tidak banyak output
        nltk.download(pkg, quiet=True)
    # Kalau gagal, beri peringatan tanpa menghentikan notebook
    except Exception as e:
        # Tampilkan paket yang gagal dan alasannya
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")
# Tanda bahwa seluruh setup sudah selesai
print("Setup selesai.")

$ git clone -b main https://github.com/william-darma-wijaya/thesis-2803.git


Cloning into 'thesis-2803'...


Repo root: /kaggle/working/thesis-2803
Branch   : main
$ pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk
Setup selesai.


## 2. Import

Semua langkah notebook ada di `src/experiments/notebook_steps.py` dan dipanggil lewat `steps`, misalnya `steps.load_models(cfg)`. Isi file itu juga diberi penjelasan per baris.

In [2]:
# sys dipakai untuk mendaftarkan lokasi kode
import sys
# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))

# Konfigurasi utama pipeline
from src.core.config import PipelineConfig
# Menentukan folder hasil per model, dan menjalankan analisis metrik serta dimensi
from src.experiments.pipeline import model_run_dirs, run_metrics_analysis, compute_dimension_analysis
# Fungsi pembuat tabel hasil
from src.experiments import result_tables as rt
# Fungsi untuk menampilkan tabel di notebook
from src.experiments.result_tables import show
# Semua langkah notebook, supaya sel di bawah tetap pendek
from src.experiments import notebook_steps as steps

# Pastikan semua import berhasil
print("Imports OK")

Imports OK


## 3. Konfigurasi

**Yang diganti tiap orang hanya `MODEL`**: `"3B"`, `"7B"`, atau `"14B"`. Hasil tiap model masuk ke `outputs/models/<model>/`, jadi tidak saling menimpa.

| Pengaturan | Arti |
|---|---|
| `SAMPLE_RATIO` | `0.05` untuk percobaan cepat (sekitar 51 pertanyaan, hasil di `outputs/trial/models/<model>/`), `1.0` untuk run final |
| `TABLE_THRESHOLD`, `COLUMN_THRESHOLD` | Skor minimum tabel dan kolom GraphRAG 2-Stage, dipilih di `trial_thresholds.ipynb` |
| `BASELINE_TABLE_THRESHOLD` | Skor minimum tabel GraphRAG 1-Stage, terpisah dari 2-Stage. Sel ini mengganti fungsi pilih tabel 1-Stage di memori notebook saja; `config.py` dan kode retrieval tidak berubah. Isi dengan `TABLE_THRESHOLD` kalau ingin sama |
| `FEW_SHOT_K` | Jumlah contoh few-shot, sama untuk kedua varian (pemenang ablation tahap 2) |
| `FEW_SHOT_SELECTION` | Cara memilih contoh few-shot, berlaku untuk kedua varian |

Pengaturan lain ada di `src/core/config.py` dan jangan diubah di sini.

In [3]:
# ===== BAGIAN YANG DIGANTI TIAP ORANG: hanya MODEL =====
# Model LLM yang dijalankan: "3B", "7B", atau "14B"
MODEL = "14B"
# Porsi dev set yang dipakai: 0.05 = percobaan cepat, 1.0 = run final skripsi
SAMPLE_RATIO = 1.0

# Skor minimum sebuah tabel supaya terpilih, GraphRAG 2-Stage (Stage 1)
TABLE_THRESHOLD = 0.60
# Skor minimum sebuah tabel supaya terpilih, GraphRAG 1-Stage (boleh beda dengan 2-Stage)
BASELINE_TABLE_THRESHOLD = 0.65
# Skor minimum sebuah kolom supaya terpilih, GraphRAG 2-Stage (Stage 2)
COLUMN_THRESHOLD = 0.70
# Cara memilih contoh few-shot: "question" atau "skeleton"
FEW_SHOT_SELECTION = "skeleton"
# Jumlah contoh few-shot, sama untuk kedua varian (pemenang ablation tahap 2)
FEW_SHOT_K = 0

# Buat konfigurasi dengan pilihan di atas
cfg = PipelineConfig(table_similarity_threshold=TABLE_THRESHOLD,
                     # Threshold kolom
                     semantic_similarity_threshold=COLUMN_THRESHOLD,
                     # Cara memilih few-shot
                     few_shot_selection=FEW_SHOT_SELECTION,
                     # Jumlah contoh few-shot
                     few_shot_k=FEW_SHOT_K)
# Ubah nama pendek ("14B") menjadi nama lengkap model
cfg.llm_model = cfg.resolve_llm_model(MODEL)
# Catat threshold tabel 1-Stage di cfg: ikut metrics_results.json dan tampilan konfigurasi
cfg.baseline_table_threshold = BASELINE_TABLE_THRESHOLD
# Hentikan notebook kalau dataset Spider tidak ditemukan
assert cfg.data_path.exists(), f"Data Spider tidak ditemukan di {cfg.data_path}. Di Kaggle, tambahkan lewat 'Add Input'."

# --- 1-Stage memakai threshold tabelnya sendiri (2-Stage tidak terpengaruh) ---
# replace() membuat salinan cfg dengan satu nilai diganti
from dataclasses import replace
# Modul retrieval 1-Stage
import src.retrieval.baseline as bl
# Fungsi pilih tabel 1-Stage yang asli; disimpan sekali supaya sel ini aman dijalankan ulang
_linking_asli = getattr(bl, "_linking_asli", bl.semantic_linking_table_level)
bl._linking_asli = _linking_asli
# Versi 1-Stage: sama persis, hanya threshold tabelnya BASELINE_TABLE_THRESHOLD
def _linking_1stage(index, question, embed_model, cfg):
    return _linking_asli(index, question, embed_model, replace(cfg, table_similarity_threshold=BASELINE_TABLE_THRESHOLD))
# Dipakai loop utama 1-Stage (run_single_baseline)
bl.semantic_linking_table_level = _linking_1stage
# Dipakai contoh satu pertanyaan (Bagian 10)
steps.semantic_linking_table_level = _linking_1stage
# Tampilkan kedua threshold tabel
print(f"Threshold tabel: 2-Stage {TABLE_THRESHOLD}, 1-Stage {BASELINE_TABLE_THRESHOLD}")

# True kalau ini run penuh, False kalau run percobaan
IS_FULL = SAMPLE_RATIO >= 1.0
# Tentukan folder hasil, raw_logs, dan QVT khusus untuk model ini
ARTIFACT_DIR, RAW_LOGS_DIR, QVT_DIR = model_run_dirs(cfg.llm_model, out_root=Path("outputs") if IS_FULL else Path("outputs/trial"))
# Run percobaan menyimpan raw_logs di folder trial supaya tidak menimpa run penuh
if not IS_FULL:
    # Arahkan raw_logs ke dalam folder hasil trial
    RAW_LOGS_DIR = ARTIFACT_DIR / "raw_logs"
# Buat folder prediksi, tabel, dan raw_logs
for folder in (ARTIFACT_DIR / "predictions", ARTIFACT_DIR / "tables", RAW_LOGS_DIR):
    # Tidak error kalau foldernya sudah ada
    folder.mkdir(parents=True, exist_ok=True)

# Kunci semua sumber acak supaya hasil run bisa diulang
steps.set_seed(cfg.seed)
# Tampilkan konfigurasi run
steps.show_run_config(cfg, SAMPLE_RATIO, ARTIFACT_DIR, RAW_LOGS_DIR)

Threshold tabel: 2-Stage 0.6, 1-Stage 0.65


Parameter,Value
Data path,/kaggle/input/datasets/alrette/spiderdataset/spider_data
Embedding model,BAAI/bge-m3
LLM model,Qwen/Qwen2.5-Coder-14B-Instruct
"Table threshold (Stage 1, GraphRAG 2-Stage)",0.6
Table threshold (GraphRAG 1-Stage),0.65
"Column threshold (Stage 2, GraphRAG 2-Stage)",0.7
Embed natural names,True
Query stopword mode,edge
Top-k tables / columns pct (0 = no cap),0% / 0%
GraphRAG 1-Stage top-k tables pct (0 = no cap),0%


## 4. Baca Data

`dev.json` berisi 1034 pasang pertanyaan dan gold SQL. Run percobaan mengambil sampel per tingkat kesulitan dengan seed tetap, jadi proporsinya sama dengan dev set penuh dan sampelnya selalu sama.

In [4]:
# Baca pertanyaan dev set; run percobaan hanya mengambil sebagian
questions = steps.load_dev_questions(cfg, SAMPLE_RATIO)
# Tulis gold SQL sesuai sampel, untuk evaluator resmi di Bagian 14
gold_path = steps.write_gold_file(questions, ARTIFACT_DIR)
# Tampilkan contoh pertanyaan pertama
print("Contoh:", questions[0]["question"])

Dev set: 1034 dari 1034 pertanyaan dipakai
Contoh: How many singers do we have?


## 5. Bangun Schema Graph

- **GraphRAG 2-Stage:** satu node mewakili satu kolom, jadi seleksi bisa sampai level kolom.
- **GraphRAG 1-Stage:** satu node mewakili satu tabel, jadi semua kolom tabel terpilih masuk prompt.

In [5]:
# Bangun graph per kolom (GraphRAG 2-Stage) dan graph per tabel (GraphRAG 1-Stage)
col_graph, tbl_graph = steps.build_graphs(cfg)

02:18:16 | INFO     | src.core.schema | Graph built: 4503 nodes, 3847 edges
02:18:17 | INFO     | src.retrieval.baseline | Table graph built: 876 table-nodes, 744 FK-edges


Graph 2-Stage: 4,503 node, 3,847 edge
Graph 1-Stage: 876 node, 744 edge


## 6. Muat Model

Dua model dimuat sekali dan dipakai kedua varian: `BAAI/bge-m3` untuk embedding, dan LLM sesuai `MODEL` (4-bit, greedy). Pastikan Accelerator = GPU.

In [6]:
# Muat model embedding dan LLM sesuai MODEL
embed_model, llm, tokenizer = steps.load_models(cfg)

02:18:17 | INFO     | sentence_transformers.base.model | No device provided, using cuda:0
02:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
02:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3 "HTTP/1.1 200 OK"
02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 307 Temporary Redirect"
02:18:17 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"


Loading embedding model: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fconfig_sentence_transformers.json=&etag=%221fba91c78a6c8e17227058ab6d4d3acb5d8630a9%22 "HTTP/1.1 200 OK"
02:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2Fconfig_sentence_transformers.json=&etag=%221fba91c78a6c8e17227058ab6d4d3acb5d8630a9%22 "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

02:18:17 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-m3.
02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/5617a9f61b028005a4858fdac845db406aefb181/README.md "HTTP/1.1 307 Temporary Redirect"
02:18:17 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2FREADME.md=&etag=%22e5a320176edd6ee1cfb256e68ee5ac0004de9447%22 "HTTP/1.1 200 OK"
02:18:17 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/README.md?%2FBAAI%2Fbge-m3%2Fresolve%2F5617a9f61b028005a4858fdac845db406aefb181%2FREADME.md=&etag=%22e5a320176edd6ee1cfb256e68ee5ac0004de9447%22 "HTTP/1.1 200 OK"


... (42 outputs left out)

02:22:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct/resolve/aedcc2d42b622764e023cf882b6652e646b95671/generation_config.json "HTTP/1.1 307 Temporary Redirect"
02:22:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-14B-Instruct/aedcc2d42b622764e023cf882b6652e646b95671/generation_config.json?%2FQwen%2FQwen2.5-Coder-14B-Instruct%2Fresolve%2Faedcc2d42b622764e023cf882b6652e646b95671%2Fgeneration_config.json=&etag=%22c28f9c697cbc09f047434efec57556505af31111%22 "HTTP/1.1 200 OK"
02:22:54 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-14B-Instruct/aedcc2d42b622764e023cf882b6652e646b95671/generation_config.json?%2FQwen%2FQwen2.5-Coder-14B-Instruct%2Fresolve%2Faedcc2d42b622764e023cf882b6652e646b95671%2Fgeneration_config.json=&etag=%22c28f9c697cbc09f047434efec57556505af31111%22 "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

02:22:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


Model siap.


## 7. Buat Index

Embedding schema dan contoh few-shot dibuat sekali di memori. Contoh few-shot hanya diambil dari `train_spider.json`, jadi tidak ada kebocoran data dev.

In [7]:
# Buat embedding schema dan index contoh few-shot
col_cache, tbl_cache, few_shot_idx = steps.build_indexes(cfg, questions, col_graph, tbl_graph, embed_model)

20 database unik di sampel ini


Index schema 2-Stage:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (54 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

few_shot_k=0 -> zero-shot untuk kedua pipeline


## 8. Siapkan Evaluator

ESM, EX, dan CM per pertanyaan dihitung oleh `src/metrics/esm_ex_cm.py` (kode custom yang membungkus parser dan evaluator resmi Spider).

In [8]:
# Siapkan alat untuk menghitung ESM, EX, CM, dan kolom gold
evaluator = steps.SqlEvaluator(cfg)
# Satukan semua alat supaya mudah diberikan ke fungsi berikutnya
tools = steps.Tools(cfg, col_graph, tbl_graph, embed_model, llm, tokenizer, col_cache, tbl_cache, few_shot_idx, evaluator)

Loading db_schema + foreign-key maps ...
db_schema: 166 DB | kmaps: 166 DB


## 9. Contoh Satu Pertanyaan: GraphRAG 2-Stage

Menampilkan tabel, kolom, DDL, contoh few-shot, dan SQL untuk satu pertanyaan. Ini hanya contoh dan **tidak masuk hasil akhir**, jadi boleh dilewati.

In [9]:
# Posisi pertanyaan yang ingin dilihat; ganti untuk melihat pertanyaan lain
WALK_IDX = 0
# Tampilkan proses 2-Stage untuk pertanyaan itu
two_stage_example = steps.walk_through_2stage(tools, questions[WALK_IDX])

DB       : concert_singer
Question : How many singers do we have?
Gold SQL : SELECT count(*) FROM singer


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Stage 1] threshold tabel 0.6 -> 3 tabel: ['singer', 'singer_in_concert', 'concert']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Stage 2] threshold kolom 0.7 -> ['singer.Singer_ID', 'singer_in_concert.Singer_ID', 'singer.Song_Name']
[Value linking] []
[Traversal] 3 jalur, mode=fk_only, total 3 kolom masuk prompt


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[Schema yang dikirim ke LLM]
CREATE TABLE singer (
    Singer_ID number PRIMARY KEY,
    Song_Name text
);

CREATE TABLE singer_in_concert (
    Singer_ID text,
    FOREIGN KEY (Singer_ID) REFERENCES singer(Singer_ID)
);


SLA kolom: P=0.00 R=0.00 F1=0.00

[Few-shot]
(kosong: few_shot_k=0)

Pred SQL : SELECT COUNT(*) FROM singer
T_in=224 T_out=8 | ESM=1 EX=1 | Difficulty=easy


## 10. Contoh Satu Pertanyaan: GraphRAG 1-Stage

Pertanyaan yang sama lewat 1-Stage: yang dipilih tabel utuh, dan semua kolomnya masuk prompt. Contoh few-shot-nya sama dengan 2-Stage.

In [10]:
# Tampilkan proses 1-Stage untuk pertanyaan yang sama, lalu bandingkan
steps.walk_through_1stage(tools, questions[WALK_IDX], two_stage_example)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Pilih tabel] threshold 0.65 -> 3 tabel: ['concert_singer.singer', 'concert_singer.singer_in_concert', 'concert_singer.concert']
[Path tracing] tabel final: ['concert_singer.singer', 'concert_singer.singer_in_concert', 'concert_singer.concert']

[Schema yang dikirim ke LLM]
CREATE TABLE singer (
    Singer_ID number PRIMARY KEY,
    Name text,
    Country text,
    Song_Name text,
    Song_release_year text,
    Age number,
    Is_male others
);

CREATE TABLE singer_in_concert (
    concert_ID number PRIMARY KEY,
    Singer_ID text,
    FOREIGN KEY (concert_ID) REFERENCES concert(concert_ID),
    FOREIGN KEY (Singer_ID) REFERENCES singer(Singer_ID)
);

CREATE TABLE concert (
    concert_ID number PRIMARY KEY,
    concert_Name text,
    Theme text,
    Stadium_ID text,
    Year text,
    FOREIGN KEY (Stadium_ID) REFERENCES stadium(Stadium_ID)
);



Variant,Predicted SQL,T_in,T_out,ESM,EX,SLA column F1
GraphRAG 1-Stage,SELECT COUNT(*) FROM singer,312,8,1,1,0.0%
GraphRAG 2-Stage,SELECT COUNT(*) FROM singer,224,8,1,1,0.0%


## 11. Jalankan Kedua Varian

Untuk tiap pertanyaan: jalankan 2-Stage dan 1-Stage, evaluasi kedua SQL, lalu simpan satu catatan per varian.

Kalau generate gagal, SQL diganti `SELECT 1` dan run tetap lanjut. Kalau evaluasi gagal, prediksi tetap tersimpan. Jumlah error ditampilkan di akhir, dan untuk run final sebaiknya 0.

In [11]:
# Jalankan 1-Stage dan 2-Stage untuk semua pertanyaan
results = steps.run_all_questions(tools, questions)
# Tampilkan jumlah pertanyaan yang diproses dan jumlah error
steps.show_loop_summary(results)

GraphRAG 1-Stage + GraphRAG 2-Stage:   0%|          | 0/1034 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[   1/1034] DB: concert_singer
  Q       : How many singers do we have?
  Gold    : SELECT count(*) FROM singer
  GraphRAG 2-Stage : SELECT COUNT(*) FROM singer
  GraphRAG 1-Stage : SELECT COUNT(*) FROM singer


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (3,147 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[1034/1034] DB: real_estate_properties
  Q       : What are the names of properties that are either houses or apartments with more than 1 room?
  Gold    : SELECT property_name FROM Properties WHERE property_type_code  =  "House" UNION SELECT property_name FROM Properties WHERE property_type_code  =  "Apartment" AND room_count  >  1
  GraphRAG 2-Stage : SELECT property_name FROM Properties WHERE property_type_code IN (SELECT property_type_code FROM Ref_Property_Types WHERE property_type_description = 'House') AND room_count > 1 UNION SELECT property_name FROM Properties WHERE property_type_code IN (SELECT property_type_code FROM Ref_Property_Types WHERE property_type_description = 'Apartment') AND room_count > 1
  GraphRAG 1-Stage : SELECT property_name FROM Properties WHERE property_type_code IN ('HSE', 'APT') AND room_count > 1


Item,Value
Samples processed,1034
raw_logs entries (per pipeline),1034
Generation errors,0
Evaluation errors,0


## 12. Amankan Prediksi dan raw_logs

File ini sengaja disimpan sekarang, sebelum hasil lain. Metrik dan dimensi membacanya, dan hasil generate yang memakan waktu lama tidak hilang kalau langkah berikutnya gagal. Hasil lain baru disimpan di Bagian 16.

In [12]:
# Simpan prediksi dan raw_logs supaya hasil generate tidak hilang kalau langkah berikutnya gagal
steps.save_generation_results(results, ARTIFACT_DIR, RAW_LOGS_DIR)

GraphRAG 2-Stage: 1034 baris prediksi dari 1034 pertanyaan -> outputs/models/Qwen2.5-Coder-14B-Instruct/predictions/predictions.txt
GraphRAG 1-Stage: 1034 baris prediksi dari 1034 pertanyaan -> outputs/models/Qwen2.5-Coder-14B-Instruct/predictions/baseline_predictions.txt
raw_logs -> data/raw_logs/Qwen2.5-Coder-14B-Instruct/ (1034 graphrag, 1034 baseline)


## 13. Hasil 6 Metrik

Tabel pertama merangkum **SLA, ESM, EX, CM, Token Consumption, dan QVT** dalam satu tabel. CM ditampilkan per klausa karena tidak boleh digabung menjadi satu angka. QVT hanya terisi pada run penuh (`SAMPLE_RATIO = 1.0`).

**TEP tidak dihitung di notebook ini.** B di rumus TEP adalah baseline skripsi DAIL-SQL, bukan GraphRAG 1-Stage, jadi TEP 1-Stage dan 2-Stage terhadap DAIL-SQL dihitung di `eval_tep.ipynb` dari hasil yang disimpan notebook ini dan `eval_dail.ipynb` (poin 45).

Selisih persentase ditulis dalam poin persen (pp). Hasil di bagian ini belum disimpan.

In [13]:
# Hitung keenam metrik dari raw_logs; hasilnya belum disimpan
metrics_results = run_metrics_analysis(RAW_LOGS_DIR, ARTIFACT_DIR / "tables", cfg.dev_json, IS_FULL, cfg, SAMPLE_RATIO, QVT_DIR, save=False)
# Tampilkan keenam metrik dalam satu tabel
show(rt.metrics_summary_table(metrics_results))

QVT grouping -- paraphrase alami dari dev.json
  Total dev rows        : 1034
  Grup (SQL identik, >=2 pertanyaan): 470
  Dev rows tercakup     : 940 (90.9%)
  Rows di luar skema (singleton, 1 pertanyaan saja): 94

-- kondisi: baseline --
  Ditulis: data/qvt_variations/Qwen2.5-Coder-14B-Instruct/baseline_qvt.json (470 query x 2 variasi)
  QVT = 0.9191  (M=383 query lolos filter, 87 dibuang -- semua variasinya EX=0)

-- kondisi: graphrag --
  Ditulis: data/qvt_variations/Qwen2.5-Coder-14B-Instruct/graphrag_qvt.json (470 query x 2 variasi)
  QVT = 0.8911  (M=381 query lolos filter, 89 dibuang -- semua variasinya EX=0)


Metric,1-stage,2-stage,Difference (2-stage − 1-stage),Note
SLA table F1,66.1%,68.9%,+2.8 pp,
SLA column F1,31.5%,53.3%,+21.8 pp,
SLA column precision,20.5%,42.5%,+22.0 pp,
SLA column recall,90.9%,89.5%,-1.5 pp,
ESM,68.6%,64.9%,-3.7 pp,
EX,74.7%,71.6%,-3.1 pp,
CM SELECT,85.8%,84.9%,-0.9 pp,
CM FROM,80.9%,80.7%,-0.2 pp,
CM WHERE,83.3%,82.1%,-1.2 pp,
CM GROUP BY,92.1%,91.5%,-0.6 pp,


### 13.1 Schema Linking Accuracy (SLA)

Precision, recall, dan F1 skema yang dikirim ke LLM dibandingkan dengan skema gold SQL. Nilainya dirata-ratakan per query.

In [14]:
# Tampilkan rincian SLA
steps.show_metric(metrics_results, "M1 SLA")

Metric 1: Schema Linking Accuracy (SLA)


Variant,Level,Population,Precision,Recall,F1
1-stage,Table,All queries,56.6%,90.9%,66.1%
2-stage,Table,All queries,59.3%,93.7%,68.9%
1-stage,Column,All queries,20.5%,90.9%,31.5%
2-stage,Column,All queries,42.5%,89.5%,53.3%
1-stage,Column,Non-empty gold schema,21.4%,94.8%,32.8%
2-stage,Column,Non-empty gold schema,44.3%,93.3%,55.6%


Difficulty,Precision 1-stage,Recall 1-stage,F1 1-stage,Precision 2-stage,Recall 2-stage,F1 2-stage
Easy,12.8%,78.2%,20.3%,33.4%,75.7%,41.3%
Medium,22.6%,95.5%,34.8%,43.7%,95.1%,56.4%
Hard,22.6%,95.0%,34.1%,48.0%,93.0%,58.2%
Extra hard,24.3%,93.3%,36.5%,47.4%,91.2%,58.0%


### 13.2 Exact Set Match (ESM)

Persentase prediksi yang struktur SQL-nya sama dengan gold.

In [15]:
# Tampilkan rincian ESM per tingkat kesulitan
steps.show_metric(metrics_results, "M2 ESM")

Metric 2: Exact Set Match (ESM)


Difficulty,1-stage,2-stage,Δ (2-stage − 1-stage)
Easy,86.7%,83.9%,-2.8 pp
Medium,70.0%,67.0%,-2.9 pp
Hard,58.0%,51.7%,-6.3 pp
Extra hard,48.8%,44.6%,-4.2 pp
All,68.6%,64.9%,-3.7 pp


### 13.3 Execution Accuracy (EX)

Persentase prediksi yang hasil eksekusinya sama dengan gold.

In [16]:
# Tampilkan rincian EX per tingkat kesulitan
steps.show_metric(metrics_results, "M3 EX")

Metric 3: Execution Accuracy (EX)


Difficulty,1-stage,2-stage,Δ (2-stage − 1-stage)
Easy,87.1%,88.3%,+1.2 pp
Medium,75.8%,71.5%,-4.3 pp
Hard,67.2%,63.8%,-3.4 pp
Extra hard,60.8%,54.8%,-6.0 pp
All,74.7%,71.6%,-3.1 pp


### 13.4 Component Match (CM)

Kecocokan per klausa SQL. Nilainya sengaja tidak digabung menjadi satu angka.

In [17]:
# Tampilkan rincian CM per klausa
steps.show_metric(metrics_results, "M4 CM")

Metric 4: Component Match (CM)


Clause,1-stage,2-stage,Δ (2-stage − 1-stage)
SELECT,85.8%,84.9%,-0.9 pp
FROM,80.9%,80.7%,-0.2 pp
WHERE,83.3%,82.1%,-1.2 pp
GROUP BY,92.1%,91.5%,-0.6 pp
ORDER BY,95.1%,93.7%,-1.4 pp
HAVING,91.3%,90.5%,-0.8 pp
Keywords,82.1%,80.8%,-1.4 pp
UNION,97.6%,97.7%,+0.1 pp
UNION ALL,–,–,–
INTERSECT,97.8%,97.9%,+0.1 pp


### 13.5 Token Consumption

T = T_in + α × T_out. TEP (B = DAIL-SQL) ada di `eval_tep.ipynb`.

In [18]:
# Tampilkan rincian Token Consumption
steps.show_metric(metrics_results, "M5 Token")

Metric 5: Token Consumption (TEP: eval_tep.ipynb)


Measure,1-stage,2-stage,Δ (2-stage − 1-stage),Relative change
Mean input tokens (T_in),320.7,272.1,-48.6,-15.1%
Mean output tokens (T_out),36.4,37.0,+0.6,+1.7%
Mean token consumption (T),430.0,383.2,-46.8,-10.9%
Median token consumption (T),414.0,366.0,-48.0,-11.6%


Difficulty,T 1-stage,T 2-stage,Δ T
Easy,358.1,299.8,-58.3
Medium,414.1,374.1,-40.0
Hard,448.4,406.3,-42.1
Extra hard,560.6,508.0,-52.6


### 13.6 Query Variance Testing (QVT)

Kestabilan jawaban pada pasangan pertanyaan parafrase. Grup yang semua variasinya salah tidak dihitung.

In [19]:
# Tampilkan rincian QVT
steps.show_metric(metrics_results, "M6 QVT")

Metric 6: Query Variance Testing (QVT)


Variant,Status,QVT,Queries included,Queries total,Reason
1-stage,ok,91.9%,383,470,
2-stage,ok,89.1%,381,470,


### 13.7 Tabel Pendukung

Rincian tambahan untuk membaca metrik, termasuk reduksi kolom dan token berpasangan per pertanyaan (arahan pembimbing, poin 33) dan latensi per tahap (lampiran, poin 37).

In [20]:
# Tampilkan tabel pendukung untuk membaca metrik
steps.show_diagnostics(results, cfg.token_output_weight)

Difficulty,Queries,EX 1-stage,EX 2-stage,Δ EX,ESM 1-stage,ESM 2-stage,Δ ESM,T 1-stage,T 2-stage,Δ T
Easy,248,87.1%,88.3%,+1.2 pp,86.7%,83.9%,-2.8 pp,358.1,299.8,-58.3
Medium,446,75.8%,71.5%,-4.3 pp,70.0%,67.0%,-2.9 pp,414.1,374.1,-40.0
Hard,174,67.2%,63.8%,-3.4 pp,58.0%,51.7%,-6.3 pp,448.4,406.3,-42.1
Extra hard,166,60.8%,54.8%,-6.0 pp,48.8%,44.6%,-4.2 pp,560.6,508.0,-52.6
All,1034,74.7%,71.6%,-3.1 pp,68.6%,64.9%,-3.7 pp,430.0,383.2,-46.8


Quantity,1-stage (mean),2-stage (mean),Reduction (1 − Σ2-stage/Σ1-stage),Median per-query reduction,Queries
Tables in prompt,2.8,2.8,+0.4%,+0.0%,1034
Columns in prompt,15.9,7.3,+53.9%,+50.0%,1034
Schema tokens (DDL only),133.1,84.5,+36.5%,+34.7%,1034
Non-schema prompt tokens (T_in − schema),187.6,187.6,+0.0%,+0.0%,1034
Prompt input tokens (T_in),320.7,272.1,+15.1%,+11.5%,1034
Output tokens (T_out),36.4,37.0,-1.7%,+0.0%,1034
Token consumption T = T_in + 3·T_out,430.0,383.2,+10.9%,+8.3%,1034


Metric,Value
Queries paired,"1,034"
EX 1-stage,74.7%
EX 2-stage,71.6%
Δ EX (2-stage − 1-stage),-3.1 pp
Schema share of 1-stage T_in (Σschema / ΣT_in),41.5%
Column reduction: table-selection effect,-4.8%
Column reduction: column-pruning effect,+56.0%
Prompts containing the whole database (1-stage),35.5%
Prompts containing the whole database (2-stage),0.0%


Difficulty,Queries,Column reduction,Schema-token reduction,T_in reduction,EX 1-stage,EX 2-stage,Δ EX
Easy,248,+62.6%,+47.8%,+18.7%,87.1%,88.3%,+1.2 pp
Medium,446,+50.0%,+33.1%,+13.2%,75.8%,71.5%,-4.3 pp
Hard,174,+56.1%,+37.2%,+16.0%,67.2%,63.8%,-3.4 pp
Extra hard,166,+50.5%,+30.6%,+14.4%,60.8%,54.8%,-6.0 pp
All,1034,+53.9%,+36.5%,+15.1%,74.7%,71.6%,-3.1 pp


Pipeline,Queries,Fallback 'SELECT 1',Empty prediction,Identical to gold (text),EX=1,ESM=1
1-stage,1034,0,0,258,772,709
2-stage,1034,0,0,247,740,671


Pipeline,EX,ESM,Queries,Share
1-stage,1,1,677,65.5%
1-stage,1,0,95,9.2%
1-stage,0,1,32,3.1%
1-stage,0,0,230,22.2%
2-stage,1,1,635,61.4%
2-stage,1,0,105,10.2%
2-stage,0,1,36,3.5%
2-stage,0,0,258,25.0%


... (2 outputs left out)

Diagnostic,Gold,1-stage,2-stage
Samples,"1,034",–,–
Queries with outer JOIN,0,1,0
Outer-JOIN predictions with ESM=1 and EX=0,–,0,0
"Official EX=0, order-insensitive row match",–,45,40
of which gold has ORDER BY,–,5,5
of which gold has no ORDER BY,–,40,35
Gold SQL not executable (excluded),0,–,–


Variant,Step,Queries,Mean (s),Median (s),P95 (s)
1-stage,Select tables,1033,0.040,0.039,0.047
1-stage,Table path tracing,1033,0.000,0.000,0.000
1-stage,Schema search (sum of the steps above),1033,0.040,0.039,0.047
1-stage,Build DDL,1033,0.000,0.000,0.000
1-stage,Total retrieval (search + DDL),1033,0.040,0.039,0.047
1-stage,Few-shot retrieval,1033,0.000,0.000,0.000
1-stage,Build prompt,1033,0.000,0.000,0.000
1-stage,Tokenize prompt for the LLM,1033,0.002,0.002,0.002
1-stage,LLM generation,1033,5.715,4.492,12.604
1-stage,Decode and clean SQL,1033,0.000,0.000,0.001


Variant,Output tokens per second,Input tokens per generation second,Retrieval share of end-to-end,Generation share of end-to-end
1-stage,6.4,56.1,0.7%,99.3%
2-stage,6.5,47.7,1.4%,98.6%


Variant,Difficulty,Step,Queries,Mean (s),Median (s),P95 (s)
1-stage,Easy,Select tables,247,0.039,0.039,0.045
1-stage,Easy,Table path tracing,247,0.000,0.000,0.000
1-stage,Easy,Schema search (sum of the steps above),247,0.039,0.039,0.045
1-stage,Easy,Build DDL,247,0.000,0.000,0.000
1-stage,Easy,Total retrieval (search + DDL),247,0.039,0.039,0.045
1-stage,Easy,Few-shot retrieval,247,0.000,0.000,0.000
1-stage,Easy,Build prompt,247,0.000,0.000,0.000
1-stage,Easy,Tokenize prompt for the LLM,247,0.002,0.001,0.002
1-stage,Easy,LLM generation,247,3.194,2.726,6.659
1-stage,Easy,Decode and clean SQL,247,0.000,0.000,0.000


## 14. Pembanding: Evaluator Resmi Spider

Menjalankan `external/spider_eval/evaluation.py` yang tidak diubah. Angka utama tetap dari Bagian 13, karena evaluator resmi gagal mem-parse outer JOIN (`IMPLEMENTATION_DECISIONS.md` poin 11).

In [21]:
# Jalankan evaluator resmi Spider sebagai pembanding; hasilnya belum disimpan
spider_scores = steps.run_spider_check(cfg, ARTIFACT_DIR, gold_path)

Pipeline,Metric,Easy,Medium,Hard,Extra hard,All
GraphRAG 2-Stage,Queries,248,446,174,166,"1,034"
GraphRAG 2-Stage,Exact match,83.9%,67.0%,51.7%,44.6%,64.9%
GraphRAG 2-Stage,Execution accuracy,88.3%,71.5%,63.8%,54.8%,71.6%
GraphRAG 1-Stage,Queries,248,446,174,166,"1,034"
GraphRAG 1-Stage,Exact match,86.7%,70.0%,58.0%,48.8%,68.6%
GraphRAG 1-Stage,Execution accuracy,87.1%,75.8%,67.2%,60.8%,74.7%


## 15. Analisis 6 Dimensi

| Dimensi | Pertanyaan yang dijawab |
|---|---|
| **1. Efisiensi** | Apakah lebih hemat token tanpa mengorbankan akurasi? (Token dan EX di sini; TEP terhadap DAIL-SQL di `eval_tep.ipynb`) |
| **2. Struktur SQL** | Bagaimana EX dan ESM berubah bersama? |
| **3. Komponen SQL** | Klausa mana yang paling terdampak? |
| **4. Robustness** | Apakah jawaban konsisten saat pertanyaan diparafrase? |
| **5. Bottleneck** | Kalau akurasi rendah, gagal di retrieval atau generation? |
| **6. Few-shot** | Berapa k yang cukup? |

- **Dimensi 4** hanya jalan pada run penuh.
- **Dimensi 5** hanya jalan kalau Dimensi 2 menemukan EX dan ESM sama-sama rendah. `ALWAYS_DIM5 = True` memaksanya jalan.
- **Dimensi 6** butuh hasil `ablation.py`. Ablation cukup dijalankan dengan satu model, jadi angkanya sama di ketiga model.

Dimensi yang datanya belum ada dilewati dengan keterangan. Hasil di bagian ini belum disimpan.

In [22]:
# Baca EX per k dari hasil ablation.py untuk Dimensi 6
EX_PER_K = steps.read_ex_per_k(cfg)
# False berarti Dimensi 5 mengikuti hasil Dimensi 2
ALWAYS_DIM5 = False
# Hitung Dimensi 1-6; hasilnya belum disimpan, dan data QVT sudah dibuat di Bagian 13
dim_results, dim_report = compute_dimension_analysis(RAW_LOGS_DIR, cfg.dev_json, IS_FULL, EX_PER_K, ALWAYS_DIM5, QVT_DIR, build_qvt=False)
# Tampilkan status dan tabel tiap dimensi
steps.show_dimensions(dim_results)

Hasil ablation belum ada: outputs/tables/ablation_ex_per_k_graphrag_skeleton.json


Dimension,Name,Status,Reason
D1,Token efficiency & performance,ok,
D2,SQL structure accuracy,ok,
D3,SQL component accuracy,ok,
D4,Robustness (QVT),ok,
D5,Retrieval vs generation (SLA × EX),ok,
D6,Few-shot ablation,skipped,"ex_per_k tidak disediakan. Jalankan ablation.py (tahap 2), lalu --ex-per-k-file outputs/tables/ablation_ex_per_k_graphrag.json (ditulis otomatis, EX in-process per k), atau manual: --ex-per-k 0=45.2,1=52.1,3=54.0,5=53.8."



Dimensi 1: Token efficiency & performance


Difficulty,Mean T 1-stage,Mean T 2-stage,Δ mean T,Relative change T,Median T 1-stage,Median T 2-stage,EX 1-stage,EX 2-stage,Δ EX
All,430.0,383.2,-46.8,-10.9%,414.0,366.0,74.7%,71.6%,-3.1 pp
Easy,358.1,299.8,-58.3,-16.3%,334.0,278.5,87.1%,88.3%,+1.2 pp
Medium,414.1,374.1,-40.0,-9.7%,412.0,373.0,75.8%,71.5%,-4.3 pp
Hard,448.4,406.3,-42.1,-9.4%,428.5,373.5,67.2%,63.8%,-3.4 pp
Extra hard,560.6,508.0,-52.6,-9.4%,521.5,468.0,60.8%,54.8%,-6.0 pp



Dimensi 2: SQL structure accuracy


Difficulty,ESM 1-stage,ESM 2-stage,Δ ESM,EX 1-stage,EX 2-stage,Δ EX
All,68.6%,64.9%,-3.7 pp,74.7%,71.6%,-3.1 pp
Easy,86.7%,83.9%,-2.8 pp,87.1%,88.3%,+1.2 pp
Medium,70.0%,67.0%,-2.9 pp,75.8%,71.5%,-4.3 pp
Hard,58.0%,51.7%,-6.3 pp,67.2%,63.8%,-3.4 pp
Extra hard,48.8%,44.6%,-4.2 pp,60.8%,54.8%,-6.0 pp


... (5 outputs left out)

Pipeline,"QVT = 1, all ESM = 1","QVT = 1, some ESM = 0","QVT < 1, all ESM = 1","QVT < 1, some ESM = 0",Share ESM issue (QVT = 1),Share ESM issue (QVT < 1),Variants missing in raw_logs
1-stage,273,48,10,52,15.0%,83.9%,0
2-stage,244,54,12,71,18.1%,85.5%,0



Dimensi 5: Retrieval vs generation (SLA × EX)


Pipeline,SLA level,EX,SLA F1,SLA precision,SLA recall,F1 ≥ 0.80,EX ≥ 0.65
1-stage,Table,74.7%,66.1%,56.6%,90.9%,No,Yes
1-stage,Column,74.7%,31.5%,20.5%,90.9%,No,Yes
2-stage,Table,71.6%,68.9%,59.3%,93.7%,No,Yes
2-stage,Column,71.6%,53.3%,42.5%,89.5%,No,Yes



Dimensi 6: Few-shot ablation (tidak ada tabel, lihat status di atas)


## 16. Simpan Semua Hasil

Setelah semua hasil tampil di atas, sel ini menyimpan semuanya ke folder model:

| File | Isi |
|---|---|
| `tables/metrics_results.json` | Keenam metrik |
| `tables/dimensions_results.json` | Hasil dimensi |
| `tables/hasil_skripsi.xlsx` | Ringkasan, rincian metrik, dimensi, dan tabel pendukung |
| `tables/spider_official_eval.csv` | Skor evaluator resmi |
| `tables/comparison_report.txt`, `tables/confound_diagnostics.json` | Proxy dan confound diagnostics |
| `tables/latency_run.json` | Waktu tahap sekali-jalan, GPU, dan puncak VRAM (sheet `Latency` dan `Latency Per Query` di Excel) |

Di Kaggle, sel ini juga membuat `hasil_skripsi_<MODEL>.xlsx` dan `hasil_<MODEL>.zip` di `/kaggle/working/` untuk diunduh.

Untuk menjalankan ulang dimensi saja tanpa generate ulang, contoh untuk 14B:

```bash
python src/experiments/run_all_dimensions.py --raw-logs-dir data/raw_logs/Qwen2.5-Coder-14B-Instruct --qvt-dir data/qvt_variations/Qwen2.5-Coder-14B-Instruct --out-dir outputs/models/Qwen2.5-Coder-14B-Instruct/tables --ex-per-k-file outputs/tables/ablation_ex_per_k_graphrag_skeleton.json
```

## 17. Menggabungkan Ketiga Model

Setelah 3B, 7B, dan 14B selesai, ekstrak zip dari tiap orang ke `outputs/models/`, lalu jalankan:

```bash
python src/experiments/combine_models.py
```

Hasilnya `outputs/tables/hasil_gabungan.xlsx` dan `hasil_gabungan.json`. Skrip ini memberi peringatan kalau ada model yang belum ada, run percobaan, atau konfigurasi yang berbeda antar model.

In [23]:
# Simpan semua hasil ke JSON dan Excel, setelah semuanya ditampilkan di atas
xlsx_path = steps.save_all_results(MODEL, results, metrics_results, spider_scores, dim_results, dim_report, ARTIFACT_DIR, RAW_LOGS_DIR)

Tersimpan: outputs/models/Qwen2.5-Coder-14B-Instruct/tables/metrics_results.json
Tersimpan: outputs/models/Qwen2.5-Coder-14B-Instruct/tables/dimensions_results.json
Tersimpan: outputs/models/Qwen2.5-Coder-14B-Instruct/tables/dimensions_report.txt
Tersimpan: outputs/models/Qwen2.5-Coder-14B-Instruct/tables/hasil_skripsi.xlsx
Sheet: Summary, M1 SLA, M2 ESM, M3 EX, M4 CM, M5 Token, M6 QVT, Dimensions, D1, D2, D3, D4, D5, Overall, By Difficulty, Token Reduction, Retrieval, SQL Diagnostics, Spider Official, Per Query, Latency, Latency Per Query
Unduh dari /kaggle/working: hasil_skripsi_14B.xlsx dan hasil_14B.zip
Waktu menulis Excel dan zip: 9.6 detik


## Catatan Desain yang Perlu Disebut di Metodologi

- **Few-shot sama untuk kedua pipeline.** GraphRAG 1-Stage dan GraphRAG 2-Stage sama-sama memakai `cfg.few_shot_k` contoh dari index yang sama (`build_few_shot_block()`), jadi selisih GraphRAG 2-Stage–GraphRAG 1-Stage murni dari retrieval. k final dipilih di ablation tahap 2 (EX tertinggi, rata-rata kedua pipeline).
- **Seleksi GraphRAG 2-Stage berbasis threshold**: tabel dan kolom disaring dengan skor cosine minimum, tanpa batas top-k%; threshold dipilih dengan SLA di `trial_thresholds.ipynb`. Langkah lain (nama natural, stopword `edge`, value linking teks, `fk_only`, sambung tabel tanpa PK) adalah **preprocessing tetap**, kontribusi masing-masing ditunjukkan rekap trial G1-G7 di notebook itu. GraphRAG 1-Stage memilih tabel dengan cara yang sama (n-gram, threshold tabel, tanpa top-k%), hanya tanpa tahap kolom: seluruh kolom tabel terpilih dikirim.
- **Traversal `fk_only`.** Dari jalur traversal hanya kolom JOIN yang masuk prompt; tabel tanpa PK disambung di graph traversal supaya JOIN lewat tabel penghubung terbaca.
- **Nama natural Spider** (`embed_natural_names`) dipakai sebagai teks embed GraphRAG 2-Stage **dan** GraphRAG 1-Stage (preprocessing data schema yang sama, jadi selisih hasil murni dari cara retrieval); anotasinya berasal dari `tables.json` Spider sendiri. SQL dan DDL tetap nama asli.
- **Few-shot skeleton** (`few_shot_selection="skeleton"`): contoh dipilih dengan kemiripan pertanyaan yang domain-nya di-mask (de-semanticization, Guo et al., 2023, disederhanakan); prompt tetap berisi pertanyaan + SQL asli contohnya. Sama untuk kedua pipeline.
- **Few-shot hanya dari `train_spider.json`.** Retrieval dari train set, bukan kebocoran dev/test.
- **Penamaan.** GraphRAG 1-Stage (`baseline` di kode) dan GraphRAG 2-Stage (`graphrag` di kode) adalah dua varian metode peneliti. Baseline skripsi adalah DAIL-SQL (`eval_dail.ipynb`); notebook ini tidak memuat perbandingan dengan baseline, dan TEP terhadap DAIL-SQL ada di `eval_tep.ipynb` (poin 38, 43, 45).
- **Proxy vs SLA.** Recall/Precision yang diprint pipeline adalah proxy name-matching, bukan SLA; SLA resmi dari `src/metrics/sla.py`.
- **ESM ≡ EM di Spider; EX order-sensitive** (`eval_exec_match()` resmi apa adanya). Diagnostik confound di Bagian 13.7 menghitung kasusnya tanpa mengubah metrik.